# Criando tabela bronze apartir de arquivo JSON

In [0]:
%sql
use catalog dbportifolio;
create database if not exists bronze;
create database if not exists silver;
create database if not exists gold;

In [0]:
%sql
create or replace table dbportifolio.bronze.livros
select * from read_files('/Workspace/Users/henrique.sza.oliveira@gmail.com/GitHub/portfolio/Databricks/files/*.json')

In [0]:
%sql
select 
    * except(endereco),
    endereco.estado,
    endereco.cidade,
    endereco.bairro,
    endereco.rua,
    endereco.numero,
    endereco.cep
from
    dbportifolio.bronze.livros

# Lendo camada bronze e gerando três tabelas pratas, uma de endereços, autores e uma de vendas de livros

In [0]:
%sql
create or replace table dbportifolio.silver.enderecos
select 
    row_number() over(order by endereco) as id,
    endereco.estado,
    endereco.cidade,
    endereco.bairro,
    endereco.rua,
    endereco.numero,
    endereco.cep
from (
    select
        distinct endereco
    from 
        dbportifolio.bronze.livros
)

In [0]:
%sql
create or replace table dbportifolio.silver.autores
select 
    row_number() over(order by autor) as id,
    autor
from (
    select
        distinct autor
    from 
        dbportifolio.bronze.livros
)

In [0]:
%sql
create or replace table dbportifolio.silver.vendas_livros
select 
    *
from (
    select
        l.* except(endereco, autor),
        a.id as autor_id,
        e.id as endereco_id
    from 
        dbportifolio.bronze.livros l
        join
            dbportifolio.silver.autores a
        on l.autor = a.autor
        join
            dbportifolio.silver.enderecos e
        on l.endereco.estado = e.estado
            and l.endereco.cidade = e.cidade
            and l.endereco.bairro = e.bairro
            and l.endereco.rua = e.rua
            and l.endereco.numero = e.numero
            and l.endereco.cep = e.cep
)

# Por fim vamos criar duas tabelas ouro, uma de autor mais vendidos e qtd de livros vendidos por cidade

In [0]:
%sql
create or replace table dbportifolio.gold.autor_mais_vendido
select 
    *
from (
    select
        a.autor,
        sum(q.quantidade) as total_vendas
    from 
        dbportifolio.silver.vendas_livros q
        join
            dbportifolio.silver.autores a
        on q.autor_id = a.id
    group by
        a.autor
)
limit 1

In [0]:
%sql
create or replace table dbportifolio.gold.vendas_cidades
select 
    c.cidade,
    sum(v.quantidade) as total_vendas
from
    dbportifolio.silver.vendas_livros v
    join
        dbportifolio.silver.enderecos c
    on v.endereco_id = c.id
group by
    c.cidade

In [0]:
%sql
drop table dbportifolio.silver.autor_mais_vendido;
drop table dbportifolio.silver.vendas_cidades